# Downscaling of a Spline with Composite Period
Let $f_{0}:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto f_{0}(x)=\sum_{k\in{\mathbb{Z}}}\,c_{0}[{k\bmod K_{0}}]\,\beta^{n_{0}}(x-\delta x_{0}-k)$ be a periodic sine-like spline at nominal scale, with a fixed, highly-composite period $K_{0}$ and with specified degree $n_{0}$ and delay $\delta_{0}.$ We corrupt the coefficients of this spline by the realization of a heavy Gaussian noise and display $f_{0}$ in thick gray. Then, we choose a positive integer minification factor $m\in{\mathbb{N}}+1$ that divides entirely $K_{0}.$ We let $f_{K_{0}\downarrow m}:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto f_{K_{0}\downarrow m}(x)=\sum_{k\in{\mathbb{Z}}}\,c_{K_{0}\downarrow m}^{n}[{k\bmod K}]\,\beta^{n}(x-\delta x-k)$ be a spline of period $K=K_{0}/m,$ arbitrary degree $n,$ arbitrary delay $\delta x,$ and spline coefficients $c_{K_{0}\downarrow m}^{n}$ chosen such that the mean-square criterion $J=\frac{1}{2}\,\int_{0}^{K_{0}}\,\left(f_{K_{0}\downarrow m}(\frac{x}{m})-f_{0}(x)\right)^{2}\,{\mathrm{d}}x$ is minimized. We display $f_{K_{0}\downarrow m}$ in blue, with samples at the integers indicated by circles and stem lines, and knots shown as black dots. The boundaries of one period are highlighted in red.

Then, we define the exact re-upscaling of $f_{K_{0}\downarrow m}$ to be $f_{\left(K_{0}\downarrow m\right)\uparrow m},$ which is a function that happens to be again at the nominal scale. We print a numeric estimate of the integral (over one period) of the product between $f_{\left(K_{0}\downarrow m\right)\uparrow m}$ and the residue $\left(f_{\left(K_{0}\downarrow m\right)\uparrow m}-f_{0}\right).$ For optimal spline coefficients $c_{K_{0}\downarrow m}^{n},$ this scalar product is expected to vanish. This is precisely what happens, up to numerical accuracy.

Finally, we print the same quantity but we avoid numeric estimates, relying instead on direct computations that are based on convolutions.

In [ ]:
# Load the required libraries
from IPython.display import display
from IPython.display import Math
import ipywidgets as widgets
import math
import matplotlib.pyplot as plt
import numpy as np
import scipy
import warnings

import splinekit as sk # This library

# Setup
period0 = 60 # Nominal period with many divisors
max_degree = 9 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay

# Minification factors
minifs = [(str(m), m) for m in range(1, period0 + 1) if 0 == period0 % m]

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Random periodic cubic spline as a corrupted sine function
f0 = sk.PeriodicSpline1D.from_spline_coeff(np.zeros(period0), degree = 3)
def update_spline_coeff (
):
    f0.spline_coeff = np.add(
        rng.standard_normal(period0),
        np.array(
            [math.sin(2.0 * np.pi * k / period0) for k in range(period0)],
            dtype = float
        )
    )
update_spline_coeff()

# Plot
def update_plot (
    degree0 = 3,
    delay0 = 0.0,
    minif = len(minifs) // 2,
    degree = 1,
    delay = 0.0,
    flipflop = True
):
    # Update of the spline
    f0.degree = degree0
    f0.delay = delay0

    # Downscaling
    fm = f0.downscaled_projected(minification = minif, degree = degree, delay = delay)

    # Dynamic range
    image = {f0.image(), fm.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    plotrange = sk.interval.Closed((
        plotrange.midpoint - 0.55 * plotrange.diameter,
        plotrange.midpoint + 0.55 * plotrange.diameter
    ))
    
    # Plots
    (fig, ax) = plt.subplots()
    # Spline at the nominal scale
    if 0 < degree0:
        # Continuous function
        x = np.linspace(-1.0, period0 // minif + 1.0, num = 600 + 1, endpoint = True)
        y = [f0.at(x0 * minif) for x0 in x]
        ax.plot(x, y, lw = 3.0, color = "#e0e0e0")
    else:
        # Function with discontinuities
        x = f0.get_knots()
        ax.plot(
            [(x[0] - 1.0) / minif, x[0] / minif],
            [f0.at(x[0] - 0.5), f0.at(x[0] - 0.5)],
            lw = 3.0,
            color = "#e0e0e0"
        )
        for (k, xk) in enumerate(x):
            if 0 < k:
                ax.plot(
                    [x[k - 1] / minif, xk / minif],
                    [f0.at(xk - 0.5), f0.at(xk - 0.5)],
                    lw = 3.0,
                    color = "#e0e0e0"
                )
        ax.plot(
            [x[-1] / minif, (x[-1] + 1.0) / minif],
            [f0.at(x[-1] + 0.5), f0.at(x[-1] + 0.5)],
            lw = 3.0,
            color = "#e0e0e0"
        )
    # Minified spline
    fm.plot((fig, ax), plotpoints = 600 + 1, plotrange = plotrange)
    # Final display
    plt.show()

    # Numeric estimate of the scalar product
    f = fm.upscaled(magnification = minif) # Magnification of the minified spline
    def integrand (
        x
    ):
        return(f.at(x) * (f.at(x) - f0.at(x)))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        djn = scipy.integrate.quad(
            integrand,
            0,
            period0,
            points = np.concatenate((f0.get_knots(), f.get_knots())),
            limit = f0.period + f.period + 1
        )
    display(Math(
        r"""
        \int_{{0}}^{{{0:}}}\,
        f_{{\left({0:}\downarrow{1:}\right)\uparrow{1:}}}(x)\,
        \left(f_{{\left({0:}\downarrow{1:}\right)\uparrow{1:}}}(x)-f_{{0}}(x))\right)\,
        {{\mathrm{{d}}}}x={2:.2E}
        """.format(period0, minif, djn[0])
    ))

    # Convolution-based scalar product
    fv = f.mirrored()
    djc = (sk.PeriodicSpline1D.convolve(fv, f).at(0) -
        sk.PeriodicSpline1D.convolve(fv, f0).at(0))
    display(Math(
        r"""
        \left(f_{{\left({0:}\downarrow{1:}\right)\uparrow{1:}}}^{{\vee}}*
        f_{{\left({0:}\downarrow{1:}\right)\uparrow{1:}}}\right)(0)-
        \left(f_{{\left({0:}\downarrow{1:}\right)\uparrow{1:}}}^{{\vee}}*
        f_{{0}}\right)(0)={2:.2E}
        """.format(period0, minif, djc)
    ))

# Interactions
flipflop_valid = widgets.Valid(value = True)
randomize_button = widgets.Button(
    description = "[ Click Me to Randomize Data ]",
    layout = widgets.Layout(width = "300px")
)
def update_spline (
    button
):
    update_spline_coeff()
    flipflop_valid.value = not flipflop_valid.value
randomize_button.on_click(update_spline)
degree0_intslider = widgets.IntSlider(
    value = 3,
    min = 0,
    max = max_degree,
    description = "degree0"
)
delay0_floatslider = widgets.FloatSlider(
    value = 0.0,
    min = -max_delay,
    max = max_delay,
    step = 0.05,
    description = "delay0"
)
minif_dropdown = widgets.Dropdown(
    options = minifs,
    value = len(minifs) // 2,
    description = "Minification"
)
degree_intslider = widgets.IntSlider(
    value = 1,
    min = 0,
    max = max_degree,
    description = "degree"
)
delay_floatslider = widgets.FloatSlider(
    value = 0.0,
    min = -max_delay,
    max = max_delay,
    step = 0.05,
    description = "delay"
)
ui = widgets.VBox([
    degree0_intslider,
    delay0_floatslider,
    minif_dropdown,
    degree_intslider,
    delay_floatslider,
    randomize_button
])
out = widgets.interactive_output(
    update_plot,
    {
        "degree0": degree0_intslider,
        "delay0": delay0_floatslider,
        "minif": minif_dropdown,
        "degree": degree_intslider,
        "delay": delay_floatslider,
        "flipflop": flipflop_valid
    }
)
display(ui, out)
